# GK-2A 12:00~14:00 Multi-year Pipeline

이 노트북은 원본 수집 상태를 확인한 뒤 tabular/long build를 수행합니다.

- Phase 1: 2019~2025 원본 GK-2A + 14:00 ASOS 수집만 수행
- 기존 Drive 원본은 재다운로드하지 않음
- 완성된 연도는 수집 자체를 SKIP
- 일부만 받은 연도는 없는 파일만 이어받음
- Phase 2: 원본 완전성 확인 후 tabular/long 변환 + TA/HM 병합
- API에서 끝내 제공되지 않는 파일은 선택적으로 NaN 처리하고 누락 CSV에 기록
- Phase 3: 2019~2025 combined CSV 생성


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 0. 설정


In [ ]:
from pathlib import Path

YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
START_MMDD = '08-24'
END_MMDD = '08-30'
START_TIME = '12:00'
END_TIME = '14:00'
STEP_MINUTES = 10
ALLOW_INCOMPLETE_BUILD = True  # 누락 NC는 NaN + missing CSV로 기록
RESUME_BUILD = True  # 완성된 연도별 build 결과는 재사용

BRANCH = 'agent/shortterm-12to14-pipeline'
REPO_DIR = Path('/content/SME_DATA')
CONFIGURED_OUTPUT_ROOT = Path('/content/drive/MyDrive/SME_DATA/processed_station_features/shortterm_12to14_data')

def source_counts(root):
    nc_count = sum(1 for _ in (root / 'raw_gk2a').glob('*/*/*/*.nc'))
    label_count = sum(1 for _ in (root / 'asos' / 'parsed').glob('asos_*.csv'))
    return nc_count, label_count

# 다른 계정·공유 드라이브에 같은 폴더가 있을 때 원본이 가장 많은 경로를 선택합니다.
candidate_patterns = [
    '/content/drive/MyDrive/*/processed_station_features/shortterm_12to14_data',
    '/content/drive/MyDrive/shortterm_12to14_data',
    '/content/drive/Shareddrives/*/SME_DATA/processed_station_features/shortterm_12to14_data',
    '/content/drive/Shareddrives/*/*/processed_station_features/shortterm_12to14_data',
]
candidates = [CONFIGURED_OUTPUT_ROOT]
for pattern in candidate_patterns:
    candidates.extend(Path('/').glob(pattern.lstrip('/')))
candidates = list(dict.fromkeys(path for path in candidates if path.exists()))
if not candidates:
    candidates = [CONFIGURED_OUTPUT_ROOT]
ranked_roots = sorted(
    ((source_counts(path), path) for path in candidates), reverse=True
)
OUTPUT_ROOT = ranked_roots[0][1]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
NC_COUNT, LABEL_COUNT = source_counts(OUTPUT_ROOT)
EXPECTED_NC = len(YEARS) * 7 * 13 * 16
EXPECTED_LABEL_FILES = len(YEARS) * 7

print('YEARS:', YEARS)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print(f'Drive source: GK2A={NC_COUNT}/{EXPECTED_NC}, ASOS={LABEL_COUNT}/{EXPECTED_LABEL_FILES}')
print('ALLOW_INCOMPLETE_BUILD:', ALLOW_INCOMPLETE_BUILD)
print('RESUME_BUILD:', RESUME_BUILD)
if NC_COUNT < EXPECTED_NC // 2 or LABEL_COUNT == 0:
    print('⚠️ 현재 마운트된 Drive에서 원본이 거의 보이지 않습니다. Phase 2 전에 계정/경로를 확인하세요.')


## 1. GitHub 최신화 + 패키지 설치


In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'fetch', 'origin'], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
!pip -q install -e .
!pip -q install requests pandas numpy xarray h5netcdf netCDF4 pyyaml

print('현재 커밋:')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


## 2. KMA API Key 설정


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    key = userdata.get('KMA_API_KEY')
except Exception:
    key = None

if not key:
    key = getpass('KMA_API_KEY 입력: ').strip()
if not key:
    raise ValueError('KMA_API_KEY가 비어 있습니다.')

os.environ['KMA_API_KEY'] = key
print('KMA_API_KEY 설정 완료')


# Phase 1 — 원본 데이터만 전부 수집

이 셀에서는 **build를 절대 실행하지 않습니다.**

연도별로 현재 Drive 원본을 먼저 검사합니다.
- 원본이 모두 있으면 `[SKIP DOWNLOAD]`
- 일부만 있으면 `[RESUME DOWNLOAD]` 후 없는 파일만 다운로드

중간에 API 제한이나 Colab 종료가 발생해도 같은 셀을 다시 실행하면 이어받습니다.


In [ ]:
import sys, subprocess

cmd = [
    sys.executable, '-u', 'scripts/shortterm_multiyear_phased.py',
    '--phase', 'collect',
    '--years', *[str(y) for y in YEARS],
    '--start-mmdd', START_MMDD,
    '--end-mmdd', END_MMDD,
    '--start-time', START_TIME,
    '--end-time', END_TIME,
    '--step-minutes', str(STEP_MINUTES),
    '--output-root', str(OUTPUT_ROOT),
]

print('[PHASE 1] 원본 수집 시작')
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## Phase 1 상태 확인

정상 기준은 연도별 GK-2A **1,456/1,456**, ASOS **7/7**입니다.
`ALL_COLLECTION_COMPLETE=False`여도 API에서 반복 실패한 파일만 남았다면 Phase 2에서 누락 허용 빌드를 선택할 수 있습니다.


In [ ]:
cmd = [
    sys.executable, '-u', 'scripts/shortterm_multiyear_phased.py',
    '--phase', 'status',
    '--years', *[str(y) for y in YEARS],
    '--start-mmdd', START_MMDD,
    '--end-mmdd', END_MMDD,
    '--start-time', START_TIME,
    '--end-time', END_TIME,
    '--step-minutes', str(STEP_MINUTES),
    '--output-root', str(OUTPUT_ROOT),
]
subprocess.run(cmd, cwd=REPO_DIR, check=True)


# Phase 2 — tabular/long 변환 + TA/HM 병합

이 단계는 2019~2025 원본 상태를 다시 검사합니다.
`ALLOW_INCOMPLETE_BUILD=True`이면 누락 NC의 채널값은 NaN으로 저장하고, `shortterm_build_missing*.csv`에 날짜·시각·채널·사유를 남깁니다. ASOS 누락도 라벨 결측으로 남습니다.

`RESUME_BUILD=True`이면 정상 완료된 연도별 결과는 재사용합니다. 다만 원본이 완성됐는데 과거 build 누락 로그가 남은 연도는 자동으로 다시 build합니다. 마지막에 combined CSV도 자동 생성합니다.


In [ ]:
NC_COUNT, LABEL_COUNT = source_counts(OUTPUT_ROOT)
print(f'[PHASE 2 PREFLIGHT] GK2A={NC_COUNT}/{EXPECTED_NC}, ASOS={LABEL_COUNT}/{EXPECTED_LABEL_FILES}')
if NC_COUNT < EXPECTED_NC // 2 or LABEL_COUNT == 0:
    raise RuntimeError(
        '현재 OUTPUT_ROOT에는 실제 원본이 거의 없습니다. ' +
        f'GK2A={NC_COUNT}/{EXPECTED_NC}, ASOS={LABEL_COUNT}/{EXPECTED_LABEL_FILES}\n' +
        '데이터를 업로드한 Google 계정으로 Drive를 다시 마운트하거나 OUTPUT_ROOT를 수정하세요.'
    )

cmd = [
    sys.executable, '-u', 'scripts/shortterm_multiyear_phased.py',
    '--phase', 'build',
    '--years', *[str(y) for y in YEARS],
    '--start-mmdd', START_MMDD,
    '--end-mmdd', END_MMDD,
    '--start-time', START_TIME,
    '--end-time', END_TIME,
    '--step-minutes', str(STEP_MINUTES),
    '--output-root', str(OUTPUT_ROOT),
]
if ALLOW_INCOMPLETE_BUILD:
    cmd.append('--allow-incomplete')
if RESUME_BUILD:
    cmd.append('--resume-build')

print('[PHASE 2] tabular/long build 시작')
print(' '.join(cmd))

# 하위 프로세스 출력을 실시간 표시하고, 실패 시 마지막 로그를 예외에 포함합니다.
from collections import deque
tail = deque(maxlen=80)
process = subprocess.Popen(
    cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    tail.append(line)
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(
        f'Phase 2 실패 (exit code {returncode}). 마지막 로그:\n' + ''.join(tail)
    )
print('[PHASE 2] 완료')


## 최종 결과 확인


In [ ]:
import pandas as pd

COMBINED = OUTPUT_ROOT / 'datasets' / 'combined'
SUMMARY = COMBINED / 'shortterm_multiyear_summary.csv'
LONG = COMBINED / 'shortterm_long_2019to2025.csv'
WIDE = COMBINED / 'shortterm_wide_2019to2025.csv'
LABELS = COMBINED / 'shortterm_labels_1400_2019to2025.csv'
MISSING = COMBINED / 'shortterm_build_missing_2019to2025.csv'

summary = pd.read_csv(SUMMARY)
display(summary)

long_df = pd.read_csv(LONG)
wide_df = pd.read_csv(WIDE)
labels_df = pd.read_csv(LABELS)

print('LONG:', long_df.shape)
print('WIDE:', wide_df.shape)
print('LABELS:', labels_df.shape)
print('Years:', sorted(wide_df['Year'].unique().tolist()))
print('Stations:', wide_df['STN_ID'].nunique())
print('TA missing:', labels_df['TA'].isna().sum())
print('HM missing:', labels_df['HM'].isna().sum())

if MISSING.exists() and MISSING.stat().st_size > 1:
    try:
        missing_df = pd.read_csv(MISSING)
    except pd.errors.EmptyDataError:
        missing_df = pd.DataFrame()
    print('Build issues:', len(missing_df))
    if len(missing_df):
        display(missing_df.head(20))
